In [36]:
import os
import certifi
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [2]:
from langchain.agents import create_react_agent,AgentExecutor

In [3]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()
GEMINI_API_KEY=os.environ.get("GEMINI_API_KEY")
TAVILY_API_KEY=os.environ.get("TAVILY_API_KEY")

In [4]:
search_tool=TavilySearchResults(max_results=3)


In [37]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [38]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'AI in Action\n\nJune 8, 2026\n\n### Google folds Display Ads into AI-first Demand Gen platform\n\nMarketing AI\n\nMay 27, 2026\n\n#### Enterprise\n\n### Anthropic deploys Claude Sonnet 5, Fable and Mythos restored\n\nInside AI\n\nJuly 1, 2026\n\n### HP accelerates enterprise workflows with OpenAI Frontier\n\nWorld of Work\n\nJune 29, 2026\n\n### Insurers pivot AI strategy toward core risk underwriting\n\nWorld of Work\n\nJune 16, 2026\n\n#### Industries\n\n### HP accelerates enterprise workflows with OpenAI Frontier\n\nWorld of Work\n\nJune 29, 2026\n\n### Omio scales travel product development using OpenAI models\n\nAI in Action\n\nJune 23, 2026\n\n### L’Oréal brings Maybelline virtual try-on to ChatGPT\n\nAI in Action\n\nJune 22, 2026\n\n#### Deep Learning\n\n### Aviva deploys AI to stop £230M in sophisticated insurance fraud [...] Entertainment & Media\n\nOctober 30, 2025\n\n### Malaysia launches Ryt Bank, its fir

In [5]:
search_tool.invoke("how is the Rao Bahabur movie?")

[{'url': 'https://en.wikipedia.org/wiki/Rao_Bahadur_(film)',
  'content': 'Rao Bahadur is an upcoming Indian Telugu-language psychological drama written, directed, and edited by Venkatesh Maha. The film stars Satyadev "Satyadev (actor)") in the titular role alongside Vikas Muppala") and Deepa Thomas"). It was presented by Mahesh Babu and Namrata Shirodkar, under GMB Entertainment"), and produced by Anurag Reddy, Sharath Chandra, and Chinta Gopalakrishna Reddy, under A+S Movies and Srichakraas Entertainments, respectively. It is set against the backdrop of a fading aristocracy that blends elements of suspense, dark comedy, and magical realism.\n\nIt is scheduled for a theatrical release on 3 July 2026.\n\n## Premise [...] ## Premise\n\nThe narrative is set against the backdrop of a fading aristocracy and follows the life of a royal figure "stuck in the past." The plot explores deep-seated psychological tension with the central tagline "Doubt is a Deadly Demon".\n\n## Cast\n\n## Producti

In [21]:
import os
from dotenv import load_dotenv

load_dotenv()
llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=os.getenv("GEMINI_API_KEY")
    )

In [22]:
llm.invoke("what is year is it?")

AIMessage(content='It is 2024.', response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-c28dc190-af4f-4c93-b4ea-44b5078aad58-0')

In [23]:
prompt=hub.pull("hwchase17/react")

c:\Users\DELL\miniconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [24]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [25]:
tools=[search_tool]

In [26]:
agent=create_react_agent(llm=llm,tools=tools,prompt=prompt)

In [31]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

In [32]:
response=agent_executor.invoke({"input": ("how is the Rao Bahabur movie?")})



> Entering new AgentExecutor chain...
Action: tavily_search_results_json
Action Input: Rao Bahabur movie review[{'url': 'https://www.greatandhra.com/movies/reviews/rao-bahadur-movie-review-a-royal-drama-that-moves-at-snail-speed', 'content': '### Rao Bahadur\n\n3 Jul 2026\n\n2.5 / 5\n\nMovie: Rao Bahadur  \n Rating: 2.5/5  \n Banner: SriChakraas Entertainments, A+S Movies, GMB Entertainment  \n Cast: Satya Dev, Vikas Muppala, Deepa Thomas, Bala Parasar, Anand Bharathi, Pranay Vaka, Kunal Kaushik, Master Kiran and others  \n Music: Smaran Sai  \n Director of Photography: Kartik Parmar  \n Production Designer: Rohan Singh  \n Producers: Anurag Reddy, Sharath Chandra, Chinta Gopalakrishna Reddy, Eshwaran Vijayaraghavan  \n Writer, Editor and Director: Venkatesh Maha  \n Release Date: July 03, 2026 [...] Skip to content\n\nGreat Andhra\n\n## Great Andhra\n\nNo.1 Telugu news website in the world | Latest Telugu News – GreatAndhra\n\nGreat Andhra\n\nHome » big story » rao bahadur movie rev

In [33]:
response["output"]

'The movie "Rao Bahadur" is an unconventional, ambitious, philosophical, and satirical film. It received ratings of 2.5/5 (Great Andhra) and 2.75/5 (Telugu360).\n\n**Key aspects of the movie include:**\n\n*   **Unique Storytelling:** It is praised for its original narrative that delves into societal prejudices related to skin color, bloodline, and caste, set between 1968 and 1991.\n*   **Strong Performances:** Satya Dev\'s acting is particularly lauded, with other cast members like Deepa Thomas and Vikas Muppala also delivering strong performances.\n*   **Technical Excellence:** The music by Smaran Sai/Samara Sai, cinematography, and production design are highlighted as top-notch, effectively creating the film\'s atmosphere and period setting.\n*   **Impactful Climax:** The ending is often described as superb and successfully ties together the film\'s thematic ambitions.\n*   **Blend of Genres:** It skillfully incorporates humor, satire, and philosophical themes.\n\n**However, it also 